In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from dotenv import load_dotenv
load_dotenv()

/Users/rahultiwari/Documents/wills_18th_july_batch/ai-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [ ]:
loader=PyPDFLoader("data_test/hr_policy_manual.pdf")
pages = loader.load()


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-31T19:53:57+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-31T19:53:57+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'data_test/hr_policy_manual.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='NovaTech Solutions Pvt. Ltd.Employee HR Policy Manual — FY 2024-25 | Version 3.2 | Confidential\nSection 1: Employment Terms1.1 Probation Period\nAll new employees at NovaTech Solutions are subject to a probation period of 6 months from their\ndate of joining. During the probation period, either party may terminate employment with a notice\nperiod of 15 days. Confirmation is subject to a satisfactory performance review by the reporting\nmanager and HR Business Partner.\n1.2 Notice Period\nPost-confirmation, the standard notice period is 60 days for all employees up to Senior Manager\nlev

In [ ]:
# install faiss before running below code

In [9]:
# pip install faiss-cpu

In [10]:
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ".", ","],
    chunk_size=512,
    chunk_overlap=100,
)

chunks = splitter.split_documents(pages)

embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(chunks, embeddings)


In [12]:
ret = vectorstore.as_retriever(search_kwargs={"k": 3})

In [17]:
ret.invoke("What are the employee benefits?", return_only_outputs=True)

[Document(id='93de8eae-0c88-467d-ade7-c05149bd097a', metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-31T19:53:57+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-31T19:53:57+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'data_test/hr_policy_manual.pdf', 'total_pages': 4, 'page': 2, 'page_label': '3'}, page_content='Section 3: Compensation and Benefits3.1 Salary Structure\nThe salary structure at NovaTech comprises the following components:\nComponent\n% of CTC\nTax Treatment\nBasic Salary\n40%\nFully Taxable\nHouse Rent Allowance (HRA)\n20%\nPartially Exempt (Sec 10(13A))\nSpecial Allowance\n20%\nFully Taxable\nPerformance Bonus\n10%\nFully Taxable\nProvident Fund (Employer)\n5%\nExempt up to limits\nGratuity (Employer)\n3%\nExempt up to Rs. 20 lakhs\nMedical Reimbursement\n2%\nExempt up to Rs. 15,000 p.a.\n3.2 Performance Appraisal Cycle'),
 Document(id='

In [15]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [21]:
prompt = ChatPromptTemplate.from_template(
       "Answer using context only.\nContext: {context}\nQuestion: {question}"
)

def fmt(docs):
    return "\n\n".join([doc.page_content for doc in docs])


# result = ret.invoke("What are the employee benefits?", return_only_outputs=True)
# print(fmt(result))

In [22]:
chain = (
    {"context": ret | fmt, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)


def ask(q):
    return chain.invoke(q), ret.invoke(q)

In [23]:
ans, src = ask("What is the notice period for senior managers?")

In [24]:
ans

'The notice period for senior managers is 60 days.'

In [26]:
# src

In [27]:
ans2, srcs2 = ask("please summarize the entire documents")


In [28]:
ans2

'The document outlines policies regarding paternity leave, confidentiality and data security, and disciplinary actions. Complaints related to bullying and discrimination must be submitted to the Internal Complaints Committee (ICC) within three months, with inquiries completed within 90 days. Employees are required to sign a Non-Disclosure Agreement (NDA) before joining, and sharing confidential data externally without legal approval can lead to termination and legal consequences. This policy supersedes all previous HR policies, and queries can be directed to hr@novatech.in or through the HR portal.'

## If I need to add all data under one vector stores

In [45]:
files =["data_test/hr_policy_manual.pdf",
        "data_test/novacrm_product_manual.pdf",
        "data_test/annual_financial_report_2024.pdf"]

all_pages = []

for pdf in files:
    loader = PyPDFLoader(pdf)
    all_pages.extend(loader.load())


splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ".", ","],
    chunk_size=512,
    chunk_overlap=100,
)


chunks = splitter.split_documents(all_pages)
vectorstore = FAISS.from_documents(chunks, embeddings)

ret_all    = vectorstore.as_retriever(search_kwargs={"k": 5})

chain_all = (
    {"context": ret_all | fmt, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)



In [46]:
q = "What was the company's performance last year and what are employee benefits? and for p1 issues what is SLA duration"
ans4  = chain_all.invoke(q)


In [47]:
print(ans4)

Last year, the company experienced a growth in employee costs, which accounted for 48% of total operating costs, reflecting a 32% headcount increase as the company scaled its engineering and customer success teams. 

Employee benefits include a salary structure with components such as Basic Salary (40% of CTC), House Rent Allowance (20%), Special Allowance (20%), Performance Bonus (10%), Provident Fund (5%), Gratuity (3%), and Medical Reimbursement (2%). 

For P1 Critical issues, the SLA duration for first response is 15 minutes, and the resolution time is 4 hours.


In [39]:
srcs4 = ret_all.invoke(q)

In [42]:
for i in srcs4:
    print(i.page_content)
    print("#####")

Medical Reimbursement
2%
Exempt up to Rs. 15,000 p.a.
3.2 Performance Appraisal Cycle
Appraisals are conducted annually in April. Process: self-assessment, manager rating, calibration.
Salary revisions effective 1st April are communicated by March 31. Mid-year promotions may be
considered in October for exceptional performers.
3.3 Employee Provident Fund (EPF)
NovaTech contributes 12% of Basic Salary towards EPF per the EPF Act, 1952. Employee
#####
Section 3: Compensation and Benefits3.1 Salary Structure
The salary structure at NovaTech comprises the following components:
Component
% of CTC
Tax Treatment
Basic Salary
40%
Fully Taxable
House Rent Allowance (HRA)
20%
Partially Exempt (Sec 10(13A))
Special Allowance
20%
Fully Taxable
Performance Bonus
10%
Fully Taxable
Provident Fund (Employer)
5%
Exempt up to limits
Gratuity (Employer)
3%
Exempt up to Rs. 20 lakhs
Medical Reimbursement
2%
Exempt up to Rs. 15,000 p.a.
3.2 Performance Appraisal Cycle
#####
Section 1: Financial Highlights 

In [30]:
len(all_pages)

13